# Basis KNN & SVM — uitleg per regel

Dit notebook toont de **basis-implementatie** op de diabetes-dataset.
- 📝 **Tekstcellen** (zoals deze) = uitleg: *wat* en *waarom*
- 💻 **Codecellen** = de code zelf, met korte commentaar (`#`)

Voer de codecellen van boven naar onder uit (Shift+Enter).

## Stap 0 — Imports

| Import | Waarom? |
|---|---|
| `pandas` | CSV inlezen als tabel (DataFrame) |
| `train_test_split` | data opsplitsen in train en test → je hebt data nodig die het model **nooit gezien** heeft |
| `StandardScaler` | alle features op dezelfde schaal → KNN en SVM rekenen met **afstanden**; anders domineert `Insulin` (tot 800) over `DiabetesPedigreeFunction` (rond 0.5) |
| `KNeighborsClassifier`, `SVC` | de twee modellen die we vergelijken (SVC = Support Vector Classifier) |
| `accuracy_score` | % juiste voorspellingen → modellen vergelijken met één getal |

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

## Stap 1 — Data inlezen

Laadt het bestand in een DataFrame `df`.
Enkel de bestandsnaam volstaat omdat de CSV in **dezelfde map** staat als dit notebook.

In [ ]:
df = pd.read_csv("diabetes.csv")
df.head()

## Stap 2 — Features (X) en target (y)

- **`X`** = de input: alle kolommen **behalve** `Outcome`. Het antwoord moet eruit, anders ziet het model de oplossing tijdens het leren.
- **`y`** = wat je wil voorspellen: diabetes ja (1) / nee (0).

Hoofdletter `X` / kleine `y` is ML-gewoonte: X is een tabel (matrix), y is één kolom (vector).

In [ ]:
X = df.drop(columns='Outcome')   # alles behalve het antwoord
y = df['Outcome']                # het antwoord

## Stap 3 — Train/test split

- Je krijgt **4 variabelen** terug, altijd in deze volgorde: `X_train, X_test, y_train, y_test`.
- `test_size=0.2` → 20% test, 80% training. Het model heeft genoeg nodig om van te leren, maar je moet ook genoeg overhouden om te testen.
- `random_state=42` → de split is willekeurig; dit getal zorgt dat je **elke keer dezelfde split** krijgt. Anders verandert je score bij elke run en kan je modellen niet eerlijk vergelijken. Op de test geven ze vaak een vaste waarde op.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Stap 4 — Schalen

1. `StandardScaler()` → maakt een lege scaler, die nog niets weet over je data.
2. `fit_transform(X_train)`:
   - **fit** = gemiddelde en standaardafwijking per kolom berekenen, **enkel op train**
   - **transform** = elke waarde omzetten naar (waarde − gemiddelde) / standaardafwijking
3. `transform(X_test)` → **enkel transform, géén fit!**
   De test moet nieuwe, onbekende data nabootsen. Fit je ook op de test, dan lekt informatie naar je model (**data leakage**) en krijg je een te mooie score.

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)   # leren + toepassen op train
X_test = scaler.transform(X_test)         # enkel toepassen op test

## Stap 5 — KNN

Elk sklearn-model volgt hetzelfde patroon: **maken → `fit` → `predict` → score**.

- `KNeighborsClassifier(n_neighbors=5)` → kijkt naar de 5 dichtste buren. Een oneven getal voorkomt een gelijke stemming. 5 is de standaardwaarde en een goed startpunt; later zoek je de beste k met GridSearch.
- `fit(X_train, y_train)` → het model leert. Bij KNN = gewoon alle punten onthouden (daarom zo snel).
- `predict(X_test)` → voorspelt 0/1 per patiënt. **Enkel `X_test`** meegeven, want het model mag het antwoord niet zien!
- `accuracy_score(y_test, y_pred)` → vergelijkt echt met voorspeld. Volgorde: **eerst echt, dan voorspeld**.

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5)   # 1. maken
knn.fit(X_train, y_train)                   # 2. trainen
y_pred_knn = knn.predict(X_test)            # 3. voorspellen
print("KNN accuracy:", accuracy_score(y_test, y_pred_knn))   # 4. score

## Stap 6 — SVM

- `SVC()` → standaardinstellingen: kernel `rbf`, `C=1`. Een goed startpunt voor niet-lineaire data.
- Daarna **exact hetzelfde patroon** als KNN.
- Dezelfde `X_train` / `X_test` gebruiken → eerlijke vergelijking: beide modellen krijgen dezelfde data.

In [ ]:
svm = SVC()                                 # 1. maken
svm.fit(X_train, y_train)                   # 2. trainen
y_pred_svm = svm.predict(X_test)            # 3. voorspellen
print("SVM accuracy:", accuracy_score(y_test, y_pred_svm))   # 4. score

## Samenvatting

1. **Inlezen** → data in Python krijgen
2. **X / y** → vragen scheiden van het antwoord
3. **Splitsen** → data apart houden om eerlijk te testen
4. **Schalen** → elke feature telt evenveel mee (fit enkel op train!)
5. **Model maken → `fit` → `predict` → score** → bij elk sklearn-model hetzelfde

### Volgende stappen
1. Onmogelijke nullen (`Glucose`, `BMI`, …) vervangen door NaN en aanvullen met `SimpleImputer`
2. Beste parameters zoeken met `GridSearchCV`
3. Ook `classification_report` bekijken, niet enkel accuracy